# Ddri Open-Meteo Weather Fetch

- 목적: 강남구 날씨 데이터를 Open-Meteo Archive API에서 추가 수집한다.
- 현재 수집 범위: `2024-01-01` 보완, `2025-01-01 ~ 2025-12-31`


In [1]:
from pathlib import Path

import pandas as pd
import requests

BASE_DIR = Path('/Users/cheng80/Desktop/ddri_work')
OUTPUT_DIR = BASE_DIR / 'works' / '02_data_collection' / '02_weather' / 'data'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LATITUDE = 37.514557
LONGITUDE = 127.0495556
BASE_URL = 'https://archive-api.open-meteo.com/v1/archive'


In [2]:
def fetch_open_meteo_history(start_date, end_date):
    params = {
        'latitude': LATITUDE,
        'longitude': LONGITUDE,
        'start_date': start_date,
        'end_date': end_date,
        'hourly': 'temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m',
        'timezone': 'Asia/Seoul',
    }
    response = requests.get(BASE_URL, params=params, timeout=60)
    response.raise_for_status()
    data = response.json()
    hourly = data['hourly']
    df = pd.DataFrame({
        'datetime': hourly['time'],
        'temperature': hourly['temperature_2m'],
        'humidity': hourly['relative_humidity_2m'],
        'precipitation': hourly['precipitation'],
        'wind_speed': hourly['wind_speed_10m'],
    })
    df['datetime'] = pd.to_datetime(df['datetime'])
    return df


In [3]:
ranges = [
    ('2024-01-01', '2024-01-01', 'ddri_weather_2024_0101_hourly.csv'),
    ('2025-01-01', '2025-12-31', 'ddri_weather_2025_hourly.csv'),
]

for start_date, end_date, output_name in ranges:
    df = fetch_open_meteo_history(start_date, end_date)
    output_path = OUTPUT_DIR / output_name
    df.to_csv(output_path, index=False)
    print(output_path, len(df))

df.head()

/Users/cheng80/Desktop/ddri_work/works/02_data_collection/02_weather/data/ddri_weather_2024_0101_hourly.csv 24


/Users/cheng80/Desktop/ddri_work/works/02_data_collection/02_weather/data/ddri_weather_2025_hourly.csv 8760


,datetime,temperature,humidity,precipitation,wind_speed
0,2025-01-01 00:00:00,-3.3,71,0.0,1.5
1,2025-01-01 01:00:00,-3.4,72,0.0,1.3
2,2025-01-01 02:00:00,-3.4,72,0.0,2.6
3,2025-01-01 03:00:00,-3.3,70,0.0,4.7
4,2025-01-01 04:00:00,-2.8,69,0.0,5.1
